# Part 5: A weather tool wrapper

**Goal:** Put a small, reliable interface around a real weather API.

**Time:** 35 minutes. **Run cells top to bottom.**

First, make sure the two small libraries used in the examples are installed. Your local environment usually already has them; Colab may need them.

In [ ]:
import importlib.util
import subprocess
import sys
missing = [pkg for module, pkg in [("requests", "requests"),
           ("dotenv", "python-dotenv")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

Find the workshop folder. In Colab, download a fresh copy if needed. This also makes `starter/` imports and file paths work in every notebook.

In [ ]:
import os
from pathlib import Path
import tempfile
root = Path.cwd()
if not (root / "starter").exists() and (root.parent / "starter").exists():
    root = root.parent
if not (root / "starter").exists():
    root = Path(tempfile.mkdtemp()) / "workshop"
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/HruthikKommuru/cwru-fall26-session7.git", str(root)])
os.chdir(root)
sys.path.insert(0, str(root))

## Where a wrapper fits

```text
model -- city name --> wrapper -- requests --> weather API
model <-- clean dict -- wrapper <-- responses -- weather API
```

A **wrapper** is a function that hides the detailed steps behind a simple input
and output. This notebook calls the wrapper directly, without a real model.

| Property | Why it helps |
| --- | --- |
| Docstring | Explains what the function does. |
| Simple typed inputs | `city: str` tells a caller to pass text. |
| Timeout | Limits the wait for each network request. |
| Retries and logging | Recovers from short failures and explains what happened. |
| Clean dictionary or clear error | Gives callers a predictable result. |

The example uses Open-Meteo's no-key public endpoints: first find the city's
coordinates, then request its current weather. Celsius is the API's default.
See [forecast docs](https://open-meteo.com/en/docs) and
[geocoding docs](https://open-meteo.com/en/docs/geocoding-api).
Credit: Open-Meteo weather data and GeoNames location data.

Load the tools and keep settings in one place. Changing a setting later lets us deliberately test timeouts and a wrong address.

In [ ]:
import requests
import logging
import time
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("workshop.weather")
WEATHER_TIMEOUT = 5
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
WAIT_SCALE = 1  # Set 0.1 for faster practice; 1 means 1s/2s/4s.

This helper makes one request, checks the HTTP status, and reads JSON. Keeping this repeated step in one place makes the main wrapper shorter.

In [ ]:
def read_json(url, params):
    response = requests.get(url, params=params, timeout=WEATHER_TIMEOUT)
    response.raise_for_status()  # Turn a bad status into a catchable error.
    return response.json()

This helper performs one full weather attempt: find coordinates, fetch weather, and return only the fields our caller needs.

In [ ]:
def weather_once(city):
    places = read_json(GEOCODING_URL, {"name": city, "count": 1})
    place = places["results"][0]  # No match is caught by the wrapper.
    params = {"latitude": place["latitude"], "longitude": place["longitude"],
              "current_weather": "true"}
    weather = read_json(WEATHER_URL, params)
    temp = weather["current_weather"]["temperature"]
    if not isinstance(temp, (int, float)) or isinstance(temp, bool):
        raise ValueError("temperature must be a number")
    return {"city": city, "temp_c": temp}

Here is the complete wrapper, using the helpers above. Three attempts and 1s/2s/4s backoff bound the work. Network and malformed-response failures become logs and an error dictionary instead of stopping the notebook.

In [ ]:
def get_weather(city: str) -> dict:
    """Return current Celsius weather, or a clear error after three attempts."""
    for attempt in range(3):
        try:
            if not isinstance(city, str) or not city.strip():
                raise ValueError("city must be nonempty text")
            return weather_once(city)
        except (requests.RequestException, ValueError, KeyError, IndexError, TypeError, AttributeError) as e:
            log.warning("try %d failed: %s", attempt + 1, e)
            time.sleep(WAIT_SCALE * 2 ** attempt)
    log.error("gave up on %s", city)
    return {"error": f"weather unavailable for {city}"}

Try a real city. A result with `error` means the service or network is unavailable here; the notebook still continues.

In [ ]:
live_weather = get_weather("Cleveland")
print(live_weather)

## Your turn — Lab 5

Write your own `get_weather_lab(city: str) -> dict`. Reuse `weather_once(city)`
or copy its steps. Include a docstring, three attempts, timeouts through
`read_json`, WARNING logs, and an ERROR log when giving up. Return the same
dictionary shapes as `get_weather`.

The settings are shared so you can break either version without rewriting it.

Replace the placeholder with your wrapper. Use the completed example as a guide, then try writing the loop without looking.

In [ ]:
get_weather_lab = None
# TODO: def get_weather_lab(city: str) -> dict:
# TODO: add a docstring and a loop with three attempts.
# TODO: call weather_once; catch network and response-shape errors.
# TODO: log warnings, wait, and finally return an error dict.

**Break it three ways:** try a made-up city, a tiny timeout, and a wrong base URL. These live results depend on your network, so we use controlled failures for the assertions below. Settings are always restored afterwards.

In [ ]:
live_breaks = []
if get_weather_lab is not None:
    saved = WEATHER_TIMEOUT, GEOCODING_URL, WAIT_SCALE
    try:
        WAIT_SCALE = 0.1
        live_breaks.append(get_weather_lab("Qzxqzxnotacity123456"))
        WEATHER_TIMEOUT = 0.001
        live_breaks.append(get_weather_lab("Cleveland"))
        WEATHER_TIMEOUT, GEOCODING_URL = 5, "https://example.invalid/search"
        live_breaks.append(get_weather_lab("Cleveland"))
    finally:
        WEATHER_TIMEOUT, GEOCODING_URL, WAIT_SCALE = saved

A tiny timeout can sometimes succeed, and some misspellings match real places. Inspect the live outcomes, then test known failures in the helper below.

In [ ]:
for label, result in zip(["made-up city", "tiny timeout", "wrong address"], live_breaks):
    print(label, result)

Provided check helper: simulate a no-match response, a timeout, and a bad address. A temporary log handler collects messages so we can check their levels.

In [ ]:
from unittest.mock import Mock, patch
def check_weather_failure(wrapper, response=None, error=None):
    records = []
    handler = logging.Handler()
    handler.emit = records.append
    log.addHandler(handler)
    try:
        with patch.object(requests, "get", return_value=response, side_effect=error):
            with patch.object(time, "sleep"):
                result = wrapper("Qzxqzxnotacity123456")
    finally:
        log.removeHandler(handler)
    assert_weather_failure(result, records)

These assertions define what a handled failure looks like: an error dictionary, three warnings, and a final error message. They are shared by all three checks.

In [ ]:
def assert_weather_failure(result, records):
    assert isinstance(result, dict) and "error" in result
    assert sum(r.levelno == logging.WARNING for r in records) == 3
    assert any(r.levelno == logging.ERROR for r in records)

**Check yourself:** these three failures must all return an error and explain it in logs. They need no internet and never depend on chance.

In [ ]:
if get_weather_lab is None:
    print("Write get_weather_lab, then run this check again.")
else:
    no_match = Mock()
    no_match.json.return_value = {"results": []}
    check_weather_failure(get_weather_lab, response=no_match)
    check_weather_failure(get_weather_lab, error=requests.Timeout("tiny timeout"))
    check_weather_failure(get_weather_lab, error=requests.ConnectionError("wrong URL"))
    print("Nice work! All three failures returned errors and useful logs.")

Also check a known successful response. Two pretend JSON responses stand in for the location search and weather response.

In [ ]:
if get_weather_lab is not None:
    location, current = Mock(), Mock()
    location.json.return_value = {"results": [{"latitude": 41.5, "longitude": -81.7}]}
    current.json.return_value = {"current_weather": {"temperature": 18.5}}
    with patch.object(requests, "get", side_effect=[location, current]) as request:
        assert get_weather_lab("Cleveland") == {"city": "Cleveland", "temp_c": 18.5}
    assert all(call.kwargs["timeout"] == 5 for call in request.call_args_list)
    print("Nice work! Success returns a clean dictionary with a timeout on both calls.")

## What you learned

- A wrapper gives a model or program a small, predictable interface.
- Two HTTP calls can become one friendly city-to-weather function.
- Controlled failure checks complement real network examples.